# Model Inference

In [1]:
!pip install lightgbm mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 83.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 81.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
import dagshub

# Add path to preprocessing script (adjust if running locally vs kaggle)
sys.path.append(os.path.abspath('.'))
sys.path.append('/kaggle/input/datasets/nikadurishvili/walmart-preprocessing')

from walmart_preprocessing import build_dataset

In [3]:
# Connect to DagsHub MLflow
dagshub.init(
    repo_owner='nikaduri',
    repo_name='ml-store-sales-forecasting',
    mlflow=True,
)

mlflow.set_tracking_uri('https://dagshub.com/nikaduri/ml-store-sales-forecasting.mlflow')

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=dceefa4d-1c31-4442-a5ed-acfa59c7e55c&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8f5473c6ecd99ff2ff7de7abf7ee5b7e5d98354668953a679e31445f8553dfaf




Accessing as nikaduri

Initialized MLflow to track repo "nikaduri/ml-store-sales-forecasting"

Repository nikaduri/ml-store-sales-forecasting initialized!

In [4]:
# Load the dataset
# BASE_PATH points to the competition data on Kaggle
BASE_PATH = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'

# Note: val_weeks doesn't matter for the test set, but we keep it consistent
ds = build_dataset(base_path=BASE_PATH, val_weeks=8, verbose=True)

Step 1/7 · Loading raw data…
  train=(421570, 5)  test=(115064, 4)  stores=(45, 3)  features=(8190, 12)
Step 2/7 · Merging tables…
  Combined shape: (536634, 17)
Step 3/7 · Creating train / val / test masks…
  Train=397,841  Val=23,729  Test=115,064
Step 4/7 · Cleaning target (train only)…
  Post-clean range: -25.9 – 283,843.0
Step 5/7 · Engineering features…
  Columns after feature engineering: 140
Step 6/7 · Cleaning features…
  Features after cleaning: 100
Step 7/7 · Selecting features…
[select_features] |r| > 0.9 vs target — review:
  store_dept_median                              r=0.9655
  roll_max_4w                                    r=0.9138
  Final feature count: 80

  X_tr=(397841, 80)  X_val=(23729, 80)  X_test=(115064, 80)
  Done ✓


In [5]:
# Load best LightGBM model from MLflow
model_name = 'best_lightgbm'
model_uri = f'models:/{model_name}/latest'

print(f'Loading model from: {model_uri}')
best_model = mlflow.lightgbm.load_model(model_uri)

Loading model from: models:/best_lightgbm/latest


In [6]:
# Make predictions on the test set
X_test = ds.X_test

print(f'Predicting for {len(X_test)} rows...')

# The model was trained on log1p(target), so we must invert it with expm1
test_pred_log = best_model.predict(X_test)
test_pred = np.expm1(test_pred_log)

Predicting for 115064 rows...


In [7]:
# Generate submission file
test_df = ds.df.loc[ds.te_mask].reset_index(drop=True)

# Create Kaggle Id format: Store_Dept_Date
test_df['Id'] = test_df['Store'].astype(str) + '_' + test_df['Dept'].astype(str) + '_' + test_df['Date'].dt.strftime('%Y-%m-%d')

# Assign predictions
test_df['Weekly_Sales'] = test_pred

# Create and save submission
submission = test_df[['Id', 'Weekly_Sales']]
submission.to_csv('submission.csv', index=False)

print('submission.csv generated successfully.')
display(submission.head())

submission.csv generated successfully.


,Id,Weekly_Sales
0,1_1_2012-11-02,35781.9385
1,1_1_2012-11-09,18939.7399
2,1_1_2012-11-16,19252.1094
3,1_1_2012-11-23,20233.4844
4,1_1_2012-11-30,23099.1760
